In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound
import os
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=1)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d')
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")


In [ ]:
## Variables de fecha como DataEntry ##
var_fecha_ini = '2026-07-01'
var_fecha_fin = '2026-07-29'
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio 1: {var_fecha_ini} ---")
print(f"--- Fecha de Fin 2: {var_fecha_fin} ---")
print(f"--- Fecha del proceso 3: {fecha_fin_dt} ---")

In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')
var_mes = fecha_fin_dt.strftime('%m')
var_fecha_file = fecha_fin_dt.strftime('%Y%m%d')

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")

In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Data/APA/t_abono_detalle/{var_anho}/{var_mes}/"
nombre_final = f"t_abono_detalle_{var_fecha_file}.csv"  # <-- CSV
proyecto = "prd-izipay-data-storage-pv"

In [ ]:
# =============================================================================
# 4. Creación de Tabla Temporal en BigQuery
# =============================================================================

temp_table_id = f"prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_csv_{var_fecha_file}"

query_temp = f"""
CREATE OR REPLACE TABLE `{temp_table_id}` AS
with aux_iden_party_data_control as (
  select
    party_id_izi,
    document_number
  from prd-izipay-data-sensitive.master_pii.iden_party_data_control
  qualify row_number() over (partition by party_id_izi order by document_number desc ) = 1
)
SELECT
  a.process_date,
  a.itc_company_id,
  a.itc_company_name,
  flujo,
  producto,
  cod_comercio,
  cod_transaccion,
  fecha_proceso,
  cod_banco,
  tipo_pago,
  AEAD.DECRYPT_STRING(b.key, a.cuenta_abono, b.constant) as cuenta_abono,
  hash_cuenta_abono,
  AEAD.DECRYPT_STRING(b.key, a.cuenta_corriente, b.constant) as cuenta_corriente,
  hash_cta_corriente,
  cod_moneda,
  importe,
  comision_abono,
  igv_comision,
  neto_1,
  cobro_devolucion,
  neto_2,
  importe_retenido,
  neto_2_dolar,
  neto_3,
  tipo_cambio,
  fecha_abono,
  tipo_cuenta,
  pago_tercero,
  ruc_tercero,
  AEAD.DECRYPT_STRING(c.key, a.nombre_tercero, c.constant) as nombre_tercero,
  tipo_doc,
  tipo_ruc,
  d.document_number as nro_documento,
  situacion,
  usuario_actualiza,
  mensaje,
  fecha_abono_mod,
  nro_dias_abono,
  sist_comp_tercero,
  cantidad,
  fac_estab,
  fecha_abono_referencial,
  nombre_cheque,
  agrupacion_abonos,
  fuerza_cuenta,
  nro_archivo_abono,
  cci,
  tipo_doc_tercero,
  cuenta_especial,
  cod_padre,
  tipo_facilitador,
  cod_estab_abono,
  nombre_comercial,
  cod_facilitador,
  tipo_doc_identificador,
  tipo_pendiente,
  fecha_entrante_DCP,
  ajuste_DCP,
  motivo_DCP,
  observacion,
  impuesto_emisor,
  cod_dcp,
  ind_capt_proces,
  filtro_abono,
  importe_solarizado,
  dq_flag_ind,
  dq_control_msg,
  dq_config_id,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.start_date as datetime)) as string) as start_date,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.end_date as datetime)) as string) as end_date,
  flag_active,
  a.record_source,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.load_date as datetime)) as string) as load_date,
  a.creation_user,
  producto_abono_det,
  tipo_abono,
  detalle_abono,
  flujo_fuente,
  flag_abono_comercio,
  des_producto,
  nombre_banco,
  des_moneda,
  des_situacion
FROM `prd-izipay-data-storage-pv.master_financial.t_abono_detalle` a
inner join prd-izipay-data-sensitive.secure_secrets.config_protected_data b on (1=1 and b.code = 'C_ACCOUNT_NUMBER')
inner join prd-izipay-data-sensitive.secure_secrets.config_protected_data c on (1=1 and c.code = 'C_BUSINESS_NAME')
left join aux_iden_party_data_control d on (d.party_id_izi = a.party_id_izi)
WHERE a.process_date >= DATE '{var_fecha_ini}'
  AND a.process_date <=  DATE '{var_fecha_fin}'
"""

print(f"Creando tabla temporal: {temp_table_id} ...")
clientBQ.query(query_temp).result()
print("✅ Tabla temporal creada correctamente.")


In [ ]:
# =============================================================================
# 5. Exportación desde Tabla Temporal a GCS (CSV)
# =============================================================================

uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{var_fecha_file}_*.csv"  # <-- CSV
print(f"Exportando desde tabla temporal a: {uri_temporal} ...")

query_export = f"""
EXPORT DATA OPTIONS (
  uri = '{uri_temporal}',
  format = 'CSV',
  overwrite = true,
  header = true
) AS
SELECT * FROM `{temp_table_id}`;
"""

clientBQ.query(query_export).result()
print("✅ Exportación a GCS completada.")

# Eliminar tabla temporal
clientBQ.delete_table(temp_table_id)
print(f"🧹 Tabla temporal eliminada: {temp_table_id}")


In [ ]:
# =============================================================================
# 6. Consolidación incremental a CSV comprimido (sin cargar todo en memoria)
# =============================================================================
# Escribe part a part en un archivo local temporal y luego sube a GCS.
# Así evitamos acumular todo el dataset en RAM.

print("Consolidando archivos CSV en uno solo (modo incremental)...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{var_fecha_file}_"

blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar.")
else:
    nombre_gz = nombre_final.replace(".csv", ".csv.gz")  # <-- extensión comprimida
    local_tmp = f"/tmp/{nombre_gz}"
    primera_parte = True

    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        df_part = pd.read_csv(uri_parte, dtype=str, keep_default_na=False)
        df_part.to_csv(
            local_tmp,
            mode='a',
            index=False,
            header=primera_parte,
            compression='gzip'                    # <-- comprime en gzip
        )
        primera_parte = False
        del df_part

    # Subir archivo consolidado a GCS
    ruta_final_full = f"{ruta_base}{nombre_gz}"   # <-- ruta con .csv.gz
    blob_final = bucket.blob(ruta_final_full)
    blob_final.upload_from_filename(local_tmp)
    os.remove(local_tmp)

    # Borrar parts temporales de GCS
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: gs://{bucket_name}/{ruta_final_full}")
    print(f"🧹 {len(blobs)} archivos temporales eliminados")